In [ ]:
# Isolate training phase
df_early_late_2_t1 <- df_early_late_2[df_early_late_2$phase == 'training_1',]
speeds_list <- unique(df_early_late_2_t1$speed_label)
set_order_list <- unique(df_early_late_2_t1$set_order)


for (speed in speeds_list) {
    
    results_storage <- list()
    raw_interaction_p_vals <- c()
    raw_trial_set_p_vals <- c()
    comparison_labels <- c()
        
    for (group in set_order_list) {
        
        df_subset <- df_early_late_2_t1[df_early_late_2_t1$speed_label == speed & df_early_late_2_t1$set_order == group, ]
        df_subset <- droplevels(df_subset) 
        
        model_anova <- aov_ez(
            id = "ppid_full",              
            dv = "flip_min_distance_xPCA_mean_bc",                  
            data = df_subset,                
            within = c("target_x_label", "trial_set"),
            fun_aggregate = mean 
        )
        
        label <- paste(speed, group)
        results_storage[[label]] <- model_anova
        
        raw_interaction_p_vals <- c(raw_interaction_p_vals, model_anova$anova_table["target_x_label:trial_set", "Pr(>F)"])
        raw_trial_set_p_vals <- c(raw_trial_set_p_vals, model_anova$anova_table["trial_set", "Pr(>F)"])
        comparison_labels <- c(comparison_labels, label)
        
        } 
            
            # Apply Holm corrections 
            adjusted_interaction_p_vals <- p.adjust(raw_interaction_p_vals, method = "holm")
            adjusted_trial_set_p_vals <- p.adjust(raw_trial_set_p_vals, method = "holm")
            
            names(adjusted_interaction_p_vals) <- comparison_labels
            names(adjusted_trial_set_p_vals) <- comparison_labels
        
            # POST-HOCS AND PRINTING
            for (label in comparison_labels) {
                
                model_anova <- results_storage[[label]]
                p_adj_inter <- adjusted_interaction_p_vals[label]
                p_adj_main <- adjusted_trial_set_p_vals[label]
                
                cat("\n========================================\n")
                cat("ANALYSIS FOR SPEED x GROUP:", label, "\n")
                cat("Holm-Adjusted Interaction p-value: ", p_adj_inter, "\n")
                cat("Holm-Adjusted trial_set Main Effect p-value: ", p_adj_main, "\n")
                cat("========================================\n")
                
                print(model_anova)
                
                # Conditional logic based on Holm-corrected results
                if (p_adj_inter <= 0.05) {
                    cat("\nSignificant Interaction. Running target-specific post-hocs:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set | target_x_label), adjust="holm"))
                    
                } else if (p_adj_main <= 0.05) {
                    cat("\nNon-Sig Interaction. Main Effect of trial_set:\n")
                    print(pairs(emmeans(model_anova, ~ trial_set)))
                    
                } else {
                    cat("\nNeither effect survived Holm correction. Skipping post-hocs.\n")
                }
            } 
            
        }